In [1]:
import torch
import torch.nn as nn


class Gemma4RMSNorm(nn.Module):
    def __init__(self, dim, eps, with_scale=True):
        super().__init__()
        self.eps = eps
        self.with_scale = with_scale

        if self.with_scale:
            self.weight = nn.Parameter(torch.ones(dim), requires_grad=True) # This is the learnable Parameter Gamma.

    def _norm(self, hidden_states):
        mean_squared = hidden_states.pow(2).mean(-1, keepdim=True) + self.eps # Mean Sqare
        return hidden_states * torch.pow(mean_squared, -0.5) # layer/root(mean_squared)

    def forward(self, hidden_states):
        normed_output = self._norm(hidden_states.float())
        if self.with_scale:
            normed_output *= self.weight.float()
        return normed_output.type_as(hidden_states)

In [2]:
toy_gemma4_rms = Gemma4RMSNorm(4,1e-6)
hidden_state_1 = torch.tensor([3,-1,4,0], dtype=torch.float16)
print(toy_gemma4_rms(hidden_state_1))

hidden_state_2 = torch.tensor([300,-100,400,0], dtype=torch.float16)
print(toy_gemma4_rms(hidden_state_2))

tensor([ 1.1768, -0.3923,  1.5693,  0.0000], dtype=torch.float16,
       grad_fn=<ToCopyBackward0>)
tensor([ 1.1768, -0.3923,  1.5693,  0.0000], dtype=torch.float16,
       grad_fn=<ToCopyBackward0>)


In [3]:
class Gemma4TextScaledWordEmbedding(nn.Embedding):
    # THis module overrides nn.Embeddings forward by multiplying with embedding scale

    def __init__(self, num_embeddings, embedding_dim, padding_idx, embed_scale=1.0):
        super().__init__(num_embeddings, embedding_dim, padding_idx)
        self.scalar_embed_scale = embed_scale
        self.register_buffer("embed_scale",torch.tensor(embed_scale), persistent=False)

    def forward(self, input_ids):
        return super().forward(input_ids) * self.embed_scale.to(self.weight.dtype)

class DiffusionGemmaTextScaledWordEmbedding(Gemma4TextScaledWordEmbedding):
    pass

In [4]:
vocab_size, dim, pad_idx = 6,4,0
embed_scale = 8.0

vanilla = nn.Embedding(vocab_size, dim, padding_idx=pad_idx) # padding will be excluded from the gradient update process

#Scaled embedding, with weights copied so we compare apples to apples
toy_gemma4_text_scaled_word_embedd = Gemma4TextScaledWordEmbedding(vocab_size,dim,pad_idx,embed_scale=embed_scale)
toy_gemma4_text_scaled_word_embedd.weight.data.copy_(vanilla.weight.data)

input_ids = torch.tensor([[1,2,3]])

vanilla_out = vanilla(input_ids)
scaled_out = toy_gemma4_text_scaled_word_embedd(input_ids)

print("Vanilla output:\n", vanilla_out, "\n")
print("Scaled output:\n", scaled_out, "\n")
print("Ratio (scaled / vanilla):\n", scaled_out/vanilla_out)

Vanilla output:
 tensor([[[ 1.2288,  0.7293,  1.8963,  0.1285],
         [-0.2533, -0.8079,  0.7371,  0.2195],
         [ 0.8924,  0.4481, -0.5686,  1.0978]]], grad_fn=<EmbeddingBackward0>) 

Scaled output:
 tensor([[[ 9.8303,  5.8342, 15.1706,  1.0279],
         [-2.0265, -6.4636,  5.8969,  1.7556],
         [ 7.1392,  3.5850, -4.5486,  8.7821]]], grad_fn=<MulBackward0>) 

Ratio (scaled / vanilla):
 tensor([[[8., 8., 8., 8.],
         [8., 8., 8., 8.],
         [8., 8., 8., 8.]]], grad_fn=<DivBackward0>)


## Rotary Positional Embedding

- 2 types: default and propositional
- Only applied to Q and K, not applied to V
- cos and sin values are precomputed and are deterministic, applying it into the Q and K happens realtime.

In [5]:
import torch
from torch import nn

# THe Gemma way of Pairing. Rotating the Half of the embedding vector.
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2]
    return torch.cat([-x2,x1], dim=-1)

# The Matrix Formula
def apply_rotary_pos_emb(x, cos, sin, unsqueeze_dim=1):
    cos=cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    return (x*cos) + (rotate_half(x) * sin)

# This is the Default RoPE calculations
def compute_default_rope_parameters(config, layer_type, device=None):
    base = config.rope_parameters[layer_type]["rope_theta"]
    dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    inv_freq = 1.0 / (
        base ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim)
    )
    return inv_freq

# This is the Proportional RoPE
def compute_proportional_rope_parameters(config, layer_type, device=None):
    rope_params = config.rope_parameters[layer_type]
    head_dim = config.global_head_dim if layer_type == "full_attention" else config.head_dim
    base = rope_params["rope_theta"]
    rope_proportion = rope_params.get("partial_rotary_factor", 1.0)# We take from the parameters or default is 1.0
    rope_angles = int(rope_proportion * head_dim // 2)

    inv_freq_rotated = 1.0 / (
        base ** (torch.arange(0, 2 * rope_angles, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / head_dim)
    )
    nope_angles = head_dim // 2 - rope_angles
    if nope_angles > 0:
        inv_freq = torch.cat([inv_freq_rotated, torch.zeros(nope_angles, dtype=torch.float32, device=device)])
    else:
        inv_freq = inv_freq_rotated
    return inv_freq


ROPE_INIT_FNS = {
    "default": compute_default_rope_parameters,
    "proportional": compute_proportional_rope_parameters,
}


class DiffusionGemmaTextRotaryEmbedding(nn.Module):
    def __init__(self, config, device=None):
        super().__init__()
        self.config = config
        # The Frequency Computation is deterministic so we use persistent=False. This means, the buffer does not get saved to state dict.
        # sliding_attention uses 'default RoPE' and full_attention uses 'proportional RoPE'
        # so we compute them separately and save them in buffers while intialising the model.
        for layer_type in set(config.layer_types):
            rope_type = config.rope_parameters[layer_type]["rope_type"]
            inv_freq = ROPE_INIT_FNS[rope_type](config, layer_type, device=device)
            self.register_buffer(f"{layer_type}_inv_freq", inv_freq, persistent=False)

    @torch.no_grad() # No gradient to be computed in RoPE, just adding positional info
    def forward(self, x, position_ids, layer_type):
        inv_freq = getattr(self, f"{layer_type}_inv_freq")# to check which rope to Use
        inv_freq_expanded = inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, -1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()

        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1,2)# Omega x m to get the unique angles
        emb = torch.cat((freqs, freqs), dim=-1)
        cos, sin = emb.cos(), emb.sin()
        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)